# Belly Button: Yelp Pipeline
**Merged notebook — Jessie + Stella**

- **Jessie**: Data collection — business search + business detail enrichment via Yelp API
- **Stella**: Data processing — recency filtering + buzz score calculation + LLM context formatter

> Note: Yelp's Reviews endpoint is unavailable on the free tier (returns 404). Individual review text and timestamps are sourced from Google Maps and Reddit instead. Yelp serves as the **restaurant discovery and metadata enrichment** layer in the pipeline.

**Pipeline flow:**
```
Yelp Business Search → Business Detail Enrichment → Recency Filter → Buzz Scoring → Format for Orchestration Layer
```

## 1. Setup

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from google.colab import userdata

API_KEY = userdata.get('yelp_api').strip()
headers = {"Authorization": f"Bearer {API_KEY}"}
print(API_KEY[:5], "...")  # confirm key loaded

z_F-E ...


## 2. Business Search - Jessie


Note: Yelp Reviews endpoint now available via Premium tier. (Apr 19th)
Real review timestamps used for buzz score calculation.


In [ ]:
def get_restaurants(location, term="restaurants", limit=20):
    response = requests.get(
        "https://api.yelp.com/v3/businesses/search",
        headers=headers,
        params={
            "term": term,
            "location": location,
            "limit": limit,
            "sort_by": "best_match"
        }
    )
    print("Status code:", response.status_code)
    data = response.json()
    if "businesses" not in data:
        print("API Error:", data)
        return pd.DataFrame()
    businesses = data["businesses"]
    results = []
    for biz in businesses:
        results.append({
            "id": biz["id"],
            "name": biz["name"],
            "rating": biz["rating"],
            "review_count": biz["review_count"],
            "address": biz["location"]["address1"],
            "categories": [c["title"] for c in biz["categories"]]
        })
    return pd.DataFrame(results)


df = get_restaurants("San Francisco", term="taco")
print(f"Fetched {len(df)} restaurants")
df.head()

Status code: 200
Fetched 20 restaurants


,id,name,rating,review_count,address,categories
0,JARsJVKLPgs_yC3cwDnp7g,La Taqueria,4.1,5108,2889 Mission St,[Mexican]
1,PgJV1wOtwTuoUz_n5Mcyxw,Tacos El Patron,4.3,1026,1500 S Van Ness Ave 100,[Tacos]
2,SGRmnarrNuVEsAjYdEoA0w,El Farolito,4.2,5733,2779 Mission St,"[Mexican, Seafood, Steakhouses]"
3,y00vIJzvt1Fveq_Um_9egw,Taqueria Vallarta,3.9,1027,3033 24th St,"[Tacos, Wine Bars, Beer Bar]"
4,FbLIxBVJAxAZPgKJsAuVSA,Tacos El Tucan,4.3,157,3600 16th St,[Tacos]


## 3. Business Detail Enrichment (Jessie) / Last edited by Stella

Enrich each restaurant with additional metadata: price range, phone, URL.

In [ ]:
def get_business_details(business_id):
    response = requests.get(
        f"https://api.yelp.com/v3/businesses/{business_id}",
        headers=headers
    )
    biz = response.json()

    # add error check
    if 'error' in biz or 'id' not in biz:
        print(f"Skipping {business_id}: {biz.get('error', 'unknown error')}")
        return None

    return {
        "id": biz["id"],
        "name": biz["name"],
        "rating": biz["rating"],
        "review_count": biz["review_count"],
        "address": biz["location"]["address1"],
        "categories": [c["title"] for c in biz["categories"]],
        "price": biz.get("price", "N/A"),
        "phone": biz.get("phone", "N/A"),
        "url": biz["url"]
    }


details = [get_business_details(bid) for bid in df["id"]]
details = [d for d in details if d is not None]  # None 제거
df_details = pd.DataFrame(details)
print(f"Enriched {len(df_details)} restaurants")
df_details.head()

Enriched 20 restaurants


,id,name,rating,review_count,address,categories,price,phone,url
0,JARsJVKLPgs_yC3cwDnp7g,La Taqueria,4.1,5108,2889 Mission St,[Mexican],$$,+14152857117,https://www.yelp.com/biz/la-taqueria-san-franc...
1,PgJV1wOtwTuoUz_n5Mcyxw,Tacos El Patron,4.3,1026,1500 S Van Ness Ave 100,[Tacos],$$,+14158297315,https://www.yelp.com/biz/tacos-el-patr%C3%B3n-...
2,SGRmnarrNuVEsAjYdEoA0w,El Farolito,4.2,5733,2779 Mission St,"[Mexican, Seafood, Steakhouses]",$$,+14158247877,https://www.yelp.com/biz/el-farolito-san-franc...
3,y00vIJzvt1Fveq_Um_9egw,Taqueria Vallarta,3.9,1027,3033 24th St,"[Tacos, Wine Bars, Beer Bar]",$$,+14159363676,https://www.yelp.com/biz/taqueria-vallarta-san...
4,FbLIxBVJAxAZPgKJsAuVSA,Tacos El Tucan,4.3,157,3600 16th St,[Tacos],$$,+14155522515,https://www.yelp.com/biz/tacos-el-tucan-san-fr...


## 4. Recency Filter (Stella)
Core to the project motivation: we only care about what's buzzing **right now**.

3-month window rationale:
- Shorter than 6 months (avoids including outdated trends)
- Longer than 1 month (avoids too few reviews given Yelp API's 3-review limit per business)

In [ ]:
def get_reviews(business_id):
    response = requests.get(
        f"https://api.yelp.com/v3/businesses/{business_id}/reviews",
        headers=headers,
        params={"limit": 3, "sort_by": "newest"}
    )
    reviews = response.json().get("reviews", [])
    return [{"business_id": business_id, "rating": r["rating"], "date": r["time_created"]} for r in reviews]

all_reviews = []
for biz_id in df_details["id"]:
    all_reviews.extend(get_reviews(biz_id))

df_reviews = pd.DataFrame(all_reviews)
df_reviews["date"] = pd.to_datetime(df_reviews["date"])

#recency filter
CUTOFF_DATE = datetime.now() - timedelta(days=90)
df_recent = df_reviews[df_reviews["date"] >= CUTOFF_DATE].copy()
print(f"Reviews before filter : {len(df_reviews)}")
print(f"Reviews after 3-month filter: {len(df_recent)}")

Reviews before filter : 54
Reviews after 3-month filter: 41


## 5. Buzz Score Calculation (Stella)

**Formula:**
```
buzz_score = 0.6 × recent_review_count_norm + 0.4 × avg_rating_norm
```
- Weights recency volume heavily (0.6) — core value prop of the project
- Both dimensions normalized 0–1 for fair comparison across restaurants
- Weights are initial estimates and will be refined through testing

In [ ]:
def compute_buzz_scores(df):
    """
    Compute buzz scores per restaurant.

    Args:
        df: DataFrame with columns [business_id, rating, date]

    Returns:
        DataFrame sorted by buzz_score descending
    """
    if len(df) == 0:
        print("⚠️  No reviews to score. Check data loading or recency filter.")
        return pd.DataFrame(columns=[
            "business_id", "recent_review_count", "avg_rating",
            "latest_review_date", "buzz_score"
        ])

    grouped = df.groupby("business_id").agg(
        recent_review_count=("rating", "count"),
        avg_rating=("rating", "mean"),
        latest_review_date=("date", "max")
    ).reset_index()

    count_range = grouped["recent_review_count"].max() - grouped["recent_review_count"].min()
    rating_range = grouped["avg_rating"].max() - grouped["avg_rating"].min()

    grouped["count_norm"] = (
        (grouped["recent_review_count"] - grouped["recent_review_count"].min())
        / (count_range + 1e-9)
    )
    grouped["rating_norm"] = (
        (grouped["avg_rating"] - grouped["avg_rating"].min())
        / (rating_range + 1e-9)
    )
    grouped["buzz_score"] = (
        0.6 * grouped["count_norm"] + 0.4 * grouped["rating_norm"]
    ).round(4)

    return grouped.sort_values("buzz_score", ascending=False)


df_buzz = compute_buzz_scores(df_recent)

# Merge back with restaurant names and details
df_final = df_buzz.merge(
    df_details[["id", "name", "categories", "price", "url"]],
    left_on="business_id", right_on="id"
).drop(columns="id")

print(f"Buzz scores computed for {len(df_final)} restaurants")
df_final[["name", "buzz_score", "avg_rating", "recent_review_count", "price"]].head(10)

Buzz scores computed for 17 restaurants


,name,buzz_score,avg_rating,recent_review_count,price
0,La Taqueria,1.0000,5.000000,3,$$
1,Al Carajo,1.0000,5.000000,3,$$
2,Ruby’s,1.0000,5.000000,3,$$
3,Suavecito Birria & Tacos,1.0000,5.000000,3,N/A
4,Cocina Mamá Cholita,0.9619,4.666667,3,$$
5,Leo's Tacos,0.9238,4.333333,3,N/A
6,El Farolito,0.9238,4.333333,3,$$
7,Underdogs Tres,0.8857,4.000000,3,$$
8,Tacos El Tucan,0.8476,3.666667,3,$$
9,Tacos El Patron,0.8476,3.666667,3,$$


## 6. Format for Orchestration Layer

---

(Stella)
Formats Yelp buzz-scored results as a clean context string for the LLM.

Mirrors Tubal's `format_as_llm_context()` for Reddit data so the orchestration layer receives **consistent input format** from all three sources (Yelp / Google Maps / Reddit).

In [ ]:
def format_yelp_as_llm_context(df_final, query, top_n=5):
    """
    Format Yelp buzz-scored restaurants as context string for the LLM.
    Mirrors Tubal's format_as_llm_context() for Reddit data.

    Args:
        df_final: merged DataFrame with buzz scores + restaurant details
        query: original user query string
        top_n: number of top restaurants to include (default 5)

    Returns:
        Formatted context string for orchestration layer
    """
    top = df_final.head(top_n).reset_index(drop=True)

    context = f'Yelp restaurant recommendations for: "{query}"\n'
    context += f'Retrieved {len(top)} restaurants (sorted by buzz score)\n'
    context += f'Scoring: 60% recent review volume + 40% average rating (3-month window)\n'
    context += '=' * 60 + '\n\n'

    for i, row in top.iterrows():
        categories = ', '.join(row['categories']) if isinstance(row['categories'], list) else row['categories']
        context += f'RESTAURANT {i + 1}\n'
        context += f'  Name: {row["name"]}\n'
        context += f'  Buzz Score: {row["buzz_score"]} (0.0–1.0, higher = more buzz)\n'
        context += f'  Avg Rating: {row["avg_rating"]:.1f} / 5.0\n'
        context += f'  Recent Reviews (3mo): {int(row["recent_review_count"])}\n'
        context += f'  Latest Activity: {str(row["latest_review_date"])[:10]}\n'
        context += f'  Categories: {categories}\n'
        context += f'  Price: {row.get("price", "N/A")}\n'
        context += f'  Source: Yelp | {row.get("url", "")}\n\n'

    return context


# Test
query = "best ice cream in San Francisco"
llm_context = format_yelp_as_llm_context(df_final, query)
print(llm_context)

Yelp restaurant recommendations for: "best ice cream in San Francisco"
Retrieved 5 restaurants (sorted by buzz score)
Scoring: 60% recent review volume + 40% average rating (3-month window)

RESTAURANT 1
  Name: La Taqueria
  Buzz Score: 1.0 (0.0–1.0, higher = more buzz)
  Avg Rating: 5.0 / 5.0
  Recent Reviews (3mo): 3
  Latest Activity: 2026-04-20
  Categories: Mexican
  Price: $$
  Source: Yelp | https://www.yelp.com/biz/la-taqueria-san-francisco-2?adjust_creative=sQ3WKo3KQiE372879oD_KA&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_lookup&utm_source=sQ3WKo3KQiE372879oD_KA

RESTAURANT 2
  Name: Al Carajo
  Buzz Score: 1.0 (0.0–1.0, higher = more buzz)
  Avg Rating: 5.0 / 5.0
  Recent Reviews (3mo): 3
  Latest Activity: 2026-03-29
  Categories: Mexican
  Price: $$
  Source: Yelp | https://www.yelp.com/biz/al-carajo-san-francisco?adjust_creative=sQ3WKo3KQiE372879oD_KA&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_lookup&utm_source=sQ3WKo3KQiE372879oD_KA

RESTAURANT 3
  Name

## 7. Save Results

In [ ]:
# Save filtered reviews
df_recent.to_csv("yelp_recent_reviews.csv", index=False)

# Save buzz scores (merged with restaurant details)
df_final.to_csv("yelp_buzz_scores.csv", index=False)

print("Saved:")
print("  - yelp_recent_reviews.csv")
print("  - yelp_buzz_scores.csv")

if len(df_final) > 0:
    top = df_final.iloc[0]
    print(f"\n🏆 Top restaurant by buzz score: {top['name']}")
    print(f"   Buzz score: {top['buzz_score']}")
    print(f"   Recent reviews: {int(top['recent_review_count'])}, Avg rating: {top['avg_rating']:.1f}")
else:
    print("\n⚠️  No buzz scores yet — will populate once real review data is connected.")

Saved:
  - yelp_recent_reviews.csv
  - yelp_buzz_scores.csv

🏆 Top restaurant by buzz score: La Taqueria
   Buzz score: 1.0
   Recent reviews: 3, Avg rating: 5.0


In [ ]:


%cd /content/belly-button
!git add .
!git commit -m "Add real Yelp reviews, fix recency filter, update formatter"
!git push origin yelp

[Errno 2] No such file or directory: '/content/belly-button'
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [ ]:
import json

test_cases = [
    "best taco in San Francisco"
]

yelp_results = {}

for i, query in enumerate(test_cases, 1):
    print(f"\nRunning TC{i}: {query}...")


    if "Berkeley" in query:
        location = "Berkeley"
        term = query.replace("best ", "").replace(" in Berkeley", "")
    else:
        location = "San Francisco"
        term = query.replace("best ", "").replace(" in San Francisco", "")

    df = get_restaurants(location, term=term, limit=20)


    all_reviews = []
    for biz_id in df["id"]:
        all_reviews.extend(get_reviews(biz_id))

    df_reviews = pd.DataFrame(all_reviews)
    df_reviews["date"] = pd.to_datetime(df_reviews["date"])

    # Recency filter
    from datetime import datetime, timedelta
    CUTOFF_DATE = datetime.now() - timedelta(days=90)
    df_recent = df_reviews[df_reviews["date"] >= CUTOFF_DATE].copy()

    # Buzz score
    df_buzz = compute_buzz_scores(df_recent)
    df_merged = df_buzz.merge(
        df[["id", "name"]],
        left_on="business_id",
        right_on="id"
    )

    # TOP 5
    top5 = df_merged["name"].head(5).tolist()
    yelp_results[query] = top5

    print(f"=== TC{i}: {query} ===")
    for j, name in enumerate(top5, 1):
        print(f"{j}. {name}")

# 저장!
with open("yelp_results.json", "w") as f:
    json.dump(yelp_results, f, indent=2)

print("\n✅ Saved to yelp_results.json!")


Running TC1: best taco in San Francisco...
Status code: 200
=== TC1: best taco in San Francisco ===
1. La Taqueria
2. Ruby’s
3. Cocina Mamá Cholita
4. Maria Isabel
5. El Farolito

✅ Saved to yelp_results.json!
